# 02. Molecular Descriptor Calculation

This notebook covers the second step in a typical QSAR workflow:
- Standardizing SMILES strings
- Calculating diverse molecular descriptors and fingerprints
- Generating a featurized dataset for modeling

ProQSAR provides built-in tools for standardization and featurization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from proqsar.data_generator import DataGenerator
from proqsar.Config.config import Config

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2.1 Load the Dataset

Load the dataset from the previous step.

In [ ]:
# Load the dataset
data_path = '../Data/testcase.csv'
df = pd.read_csv(data_path)

print(f"Dataset loaded: {len(df)} compounds")
df.head()

## 2.2 Configure Featurizer

ProQSAR supports various molecular descriptors and fingerprints:
- **Fingerprints**: ECFP (Morgan), RDK, MACCS, Avalon, AtomPair, TopologicalTorsion
- **Descriptors**: RDKit 2D descriptors, Mordred descriptors

You can specify which features to calculate in the Config.

In [ ]:
# Configure the featurizer
# Example: Calculate ECFP4 and RDK5 fingerprints
config = Config(
    featurizer={
        "feature_types": ["ECFP4", "RDK5", "MACCS"],
        # You can also add descriptors:
        # "feature_types": ["ECFP4", "RDK5", "RDKit2D"],
    }
)

print("Featurizer configured successfully!")
print(f"Features to calculate: {config.featurizer_config['feature_types']}")

## 2.3 Initialize DataGenerator

The DataGenerator handles SMILES standardization and feature calculation.

In [ ]:
# Initialize DataGenerator
data_generator = DataGenerator(
    activity_col="pChEMBL",
    id_col="Smiles",
    smiles_col="Smiles",
    config=config,
    save_dir="../Project/DataGenerator",
    data_name="qsar_example"
)

print("DataGenerator initialized successfully!")

## 2.4 Generate Features

Standardize SMILES and calculate molecular descriptors.

In [ ]:
# Generate features
print("Standardizing SMILES and calculating descriptors...")
feature_df = data_generator.generate(df)

print(f"\nFeature generation complete!")
print(f"Original dataset shape: {df.shape}")
print(f"Featurized dataset shape: {feature_df.shape}")
print(f"Number of molecular descriptors: {feature_df.shape[1] - 2}")  # Subtract ID and activity columns

In [ ]:
# Display first few rows
print("\nFirst few rows of featurized data:")
feature_df.head()

## 2.5 Analyze Feature Statistics

Examine the distribution and characteristics of calculated features.

In [ ]:
# Get feature columns (exclude ID and activity)
feature_cols = [col for col in feature_df.columns if col not in ['Smiles', 'pChEMBL']]

print(f"Total number of features: {len(feature_cols)}")
print(f"\nFeature statistics:")
feature_df[feature_cols].describe()

In [ ]:
# Check for constant features (zero variance)
feature_variance = feature_df[feature_cols].var()
constant_features = feature_variance[feature_variance == 0]

print(f"Number of constant features: {len(constant_features)}")
if len(constant_features) > 0:
    print("\nConstant features (will be removed during preprocessing):")
    print(constant_features.index.tolist())

In [ ]:
# Plot feature variance distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
feature_variance[feature_variance > 0].hist(bins=50, edgecolor='black')
plt.xlabel('Variance')
plt.ylabel('Frequency')
plt.title('Distribution of Feature Variance')
plt.yscale('log')

plt.subplot(1, 2, 2)
feature_means = feature_df[feature_cols].mean()
plt.scatter(feature_means, feature_variance, alpha=0.5)
plt.xlabel('Feature Mean')
plt.ylabel('Feature Variance')
plt.title('Feature Mean vs Variance')
plt.yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Check for missing values in features
missing_values = feature_df[feature_cols].isnull().sum()
features_with_missing = missing_values[missing_values > 0]

print(f"Features with missing values: {len(features_with_missing)}")
if len(features_with_missing) > 0:
    print("\nFeatures with missing values:")
    print(features_with_missing)

## 2.6 Save Featurized Dataset

Save the featurized dataset for use in subsequent notebooks.

In [ ]:
# Save to CSV
output_path = '../Project/featurized_data.csv'
feature_df.to_csv(output_path, index=False)

print(f"Featurized dataset saved to: {output_path}")
print(f"Shape: {feature_df.shape}")

## 2.7 Summary

Summarize the descriptor calculation process.

In [ ]:
print("="*60)
print("DESCRIPTOR CALCULATION SUMMARY")
print("="*60)
print(f"Input compounds: {len(df)}")
print(f"Output compounds: {len(feature_df)}")
print(f"Total features calculated: {len(feature_cols)}")
print(f"Constant features: {len(constant_features)}")
print(f"Features with missing values: {len(features_with_missing)}")
print(f"\nFeature types: {config.featurizer_config['feature_types']}")
print(f"\nFeaturized data saved to: {output_path}")
print("\nDataset is ready for feature selection!")
print("="*60)

## Next Steps

The molecular descriptors have been calculated. The next notebook (03_feature_selection.ipynb) will:
- Load the featurized dataset
- Apply feature selection techniques
- Identify the most relevant descriptors for QSAR modeling